# AIND Ephys Spike Sorting (Kilosort4)

Build an `AINDEPhysSpikesortKilosort4ScanConfig`, expand it with `GridScanGenerationTask`, and run the `AINDEPhysSpikesortKilosort4Task` for each coordinate. The task itself clones [aind-ephys-spikesort-kilosort4](https://github.com/AllenNeuralDynamics/aind-ephys-spikesort-kilosort4) on first use, seeds its `data/` with the `preprocessed_<name>/` and `binary_<name>.json` produced by `AINDEPhysPreprocessingTask`, writes a `params_obi.json`, runs `code/run_capsule.py`, and copies the resulting `spikesorted_<name>/` into the single config's `coordinate_output_root`.

Kilosort4 fits whitening / template models from the data and refuses to run on a 1-second / few-channel slice. So this notebook **re-runs preprocessing at 8 s** (a fresh `AINDEPhysPreprocessingScanConfig` with `t_stop=8.0`) before invoking the sorter — and tunes a few sorter knobs (`whitening_range`, `nskip`, `nearest_chans`, `nearest_templates`) for the small toy probe.

Outputs: preprocessing → `obi-output/03_aind_ephys_spikesort_kilosort4/preprocessing/0/`, KS4 → `obi-output/03_aind_ephys_spikesort_kilosort4/sorting/0/`.

## Imports and prerequisites

In [1]:
import subprocess
import sys
from pathlib import Path

import obi_one as obi
from obi_one.scientific.tasks.aind_ephys._02_preprocessing.blocks import MotionCorrection
from obi_one.scientific.tasks.aind_ephys._03_kilosort4.blocks import Kilosort4Sorter

In [2]:
subprocess.run(
    [
        "uv", "pip", "install", "--python", sys.executable,
        "kilosort", "aind-data-schema", "scikit-learn",
    ],
    check=True,
)

Using Python 3.12.9 environment at: /Users/james/Documents/obi/code/obi-main/obi-one/.venv


Resolved 44 packages in 577ms
Uninstalled 3 packages in 53ms
Installed 3 packages in 20ms
 - pydantic==2.12.5
 + pydantic==2.11.10
 - pydantic-core==2.41.5
 + pydantic-core==2.33.2
 - setuptools==82.0.0
 + setuptools==81.0.0


CompletedProcess(args=['uv', 'pip', 'install', '--python', '/Users/james/Documents/obi/code/obi-main/obi-one/.venv/bin/python', 'kilosort', 'aind-data-schema', 'scikit-learn'], returncode=0)

## Re-run preprocessing on an 8-second window

Reads `job_*.json` from notebook 01's coord 0 and writes the preprocessed recording into `obi-output/03_aind_ephys_spikesort_kilosort4/preprocessing/0/`.

In [3]:
dispatch_output_path = (
    Path.cwd() / "../../../obi-output/01_aind_ephys_dispatch/grid_scan/0"
).resolve()
assert dispatch_output_path.exists(), (
    f"{dispatch_output_path} not found. Run 01_aind_ephys_dispatch.ipynb first."
)

preproc_scan = obi.AINDEPhysPreprocessingScanConfig(
    initialize=obi.AINDEPhysPreprocessingScanConfig.Initialize(
        dispatch_output_path=dispatch_output_path,
        denoising_strategy="cmr",
        filter_type="highpass",
        min_preprocessing_duration=0.5,
        t_start=0.0,
        t_stop=8.0,
        n_jobs=1,
    ),
    motion_correction=MotionCorrection(compute=False, apply=False),
)

preproc_grid = obi.GridScanGenerationTask(
    form=preproc_scan,
    output_root="../../../obi-output/03_aind_ephys_spikesort_kilosort4/preprocessing",
    coordinate_directory_option="ZERO_INDEX",
)
preproc_grid.execute()
obi.run_tasks_for_generated_scan(preproc_grid)

preproc_output = Path(preproc_grid.single_configs[0].coordinate_output_root)
print("Preprocessing output:", preproc_output)

[2026-04-29 18:34:06,757] INFO: Seeded 1 job file(s) into /tmp/aind-ephys-preprocessing/data


[2026-04-29 18:34:06,758] INFO: Running python -u run_capsule.py --params params_obi.json --motion skip --n-jobs 1 --t-start 0.0 --t-stop 8.0


[2026-04-29 18:34:10,723] INFO: Running preprocessing with the following parameters:
	DENOISING_STRATEGY: cmr
	FILTER TYPE: highpass
	REMOVE_OUT_CHANNELS: True
	REMOVE_BAD_CHANNELS: True
	MAX BAD CHANNEL FRACTION: 0.5
	COMPUTE_MOTION: False
	APPLY_MOTION: False
	MOTION PRESET: dredge_fast
	MOTION TEMPORAL BIN S: 1
	T_START: 0.0
	T_STOP: 8.0
	MIN_DURATION FOR PREPROCESSING: 0.5
	N_JOBS: 1
Found 1 configurations


PREPROCESSING
Preprocessing recording: session0 - block0_None_recording1
	Original recording duration: 10.0 -- Clipping to 0.0-8.0 s
	Duration: 8.0 s
	Bad channel detection:
		- dead channels - 0
		- noise channels - 0
		- out channels - 0
	Removing 0 out channels
	Removing 0 channels after cmr preprocessing
write_binary_recording 
engine=process - n_jobs=1 - samples_per_chunk=30,000 - chunk_memory=8.01 MiB - total_memory=8.01 MiB - chunk_duration=1.00s
PREPROCESSING time: 3.5s



Preprocessing output: ../../../obi-output/03_aind_ephys_spikesort_kilosort4/preprocessing/0


## Build the spike-sorting scan config

In [4]:
ks_scan = obi.AINDEPhysSpikesortKilosort4ScanConfig(
    initialize=obi.AINDEPhysSpikesortKilosort4ScanConfig.Initialize(
        preprocessing_output_path=preproc_output,
        skip_motion_correction=True,
        min_drift_channels=64,
        n_jobs=1,
    ),
    sorter=Kilosort4Sorter(
        do_correction=False,
        nblocks=0,
        torch_device="cpu",
        whitening_range=16,
        nskip=1,
        nearest_chans=8,
        nearest_templates=16,
    ),
)

## Generate the grid scan and run the sorting task

In [5]:
ks_grid = obi.GridScanGenerationTask(
    form=ks_scan,
    output_root="../../../obi-output/03_aind_ephys_spikesort_kilosort4/sorting",
    coordinate_directory_option="ZERO_INDEX",
)
ks_grid.execute()

obi.run_tasks_for_generated_scan(ks_grid)

[2026-04-29 18:34:10,773] INFO: Seeded 1 preprocessed recording(s) into /tmp/aind-ephys-spikesort-kilosort4/data


[2026-04-29 18:34:10,774] INFO: Running python -u run_capsule.py --params params_obi.json --n-jobs 1 --min-drift-channels 64 --skip-motion-correction --raise-if-fails


[2026-04-29 18:34:27,484] INFO: 

SPIKE SORTING WITH KILOSORT4

	RAISE_IF_FAILS: True
	SKIP_MOTION_CORRECTION: True
	MIN_DRIFT_CHANNELS: 64
	N_JOBS: -1
Sorting recording: block0_None_recording1
Loading recording from binary JSON
BinaryFolderRecording: 70 channels - 30.0kHz - 1 segments - 240,001 samples - 8.00s 
                       float32 dtype - 64.09 MiB
Drift correction disabled
	Raw sorting output: KiloSortSortingExtractor: 14 units - 1 segments - 30.0kHz
	Sorting output without empty units: UnitsSelectionSorting: 14 units - 1 segments - 30.0kHz
	Saving results to ../results/spikesorted_block0_None_recording1
SPIKE SORTING time: 15.85s



[PosixPath('../../../obi-output/03_aind_ephys_spikesort_kilosort4/sorting/0')]

## Load the sorting result

In [6]:
import spikeinterface.full as si

coord_dir = Path(ks_grid.single_configs[0].coordinate_output_root)
print("coordinate_output_root:", coord_dir)

sorting_dirs = [
    p for p in coord_dir.iterdir() if p.is_dir() and p.name.startswith("spikesorted_")
]
print("Sorting dirs:", [p.name for p in sorting_dirs])

if sorting_dirs:
    sorting = si.load(sorting_dirs[0])
    print(sorting)
    print("num_units:", len(sorting.unit_ids))
    print("unit_ids:", list(sorting.unit_ids))

coordinate_output_root: ../../../obi-output/03_aind_ephys_spikesort_kilosort4/sorting/0
Sorting dirs: ['spikesorted_block0_None_recording1']
NumpyFolder (NumpyFolderSorting): 14 units - 1 segments - 30.0kHz
num_units: 14
unit_ids: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13)]
